# Run your six-language X-Voice fine-tune in Google Colab

This notebook runs **Stage 1 inference** with the fine-tuned checkpoint. It uses the six added language IDs in this repository (`bd`, `mn`, `st`, `ck`, `gr`, `ng`) and the matching expanded `vocab.txt`. Use the checkpoint produced by fine-tuning, not the original base checkpoint.

You need your final checkpoint (`model_last.pt` or a numbered `.pt`/`.safetensors` file), the exact expanded `vocab.txt` used for fine-tuning, and a reference voice recording. Put them in Google Drive, then choose **Runtime → Change runtime type → GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit these paths for your Drive files.
CHECKPOINT = '/content/drive/MyDrive/xvoice/model_last.pt'
VOCAB = '/content/drive/MyDrive/xvoice/vocab.txt'
REFERENCE_AUDIO = '/content/drive/MyDrive/xvoice/reference.wav'

In [ ]:
import os, torch
assert os.path.isfile(CHECKPOINT), f'Missing checkpoint: {CHECKPOINT}'
assert os.path.isfile(VOCAB), f'Missing expanded vocab: {VOCAB}'
assert os.path.isfile(REFERENCE_AUDIO), f'Missing reference audio: {REFERENCE_AUDIO}'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Enable a GPU in Runtime → Change runtime type.'

In [ ]:
%cd /content
!if [ -d /content/X-Voice/.git ]; then git -C /content/X-Voice pull; else git clone https://github.com/lavishchauhan5555/x-voice.git /content/X-Voice; fi
%cd /content/X-Voice
!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng
!python -m pip install -q -e .

## Choose text and language

Set `GEN_LANG` to one of the six added IDs: Bolo `bd`, Manipuri `mn`, Santali `st`, Chakma `ck`, Garo `gr`, or Nagamese `ng`. You may also choose a base language supported by the model, such as `en`. The reference transcript must match the speech in the reference audio.

In [ ]:
REF_TEXT = 'Write the exact words spoken in your reference recording here.'
REF_LANG = 'en'
GEN_TEXT = 'Enter the sentence you want the model to speak.'
GEN_LANG = 'bd'  # bd, mn, st, ck, gr, ng, or a supported base language
OUTPUT_DIR = '/content/xvoice_output'
OUTPUT_FILE = 'generated.wav'

In [ ]:
import yaml
from pathlib import Path

# Preserve the base model architecture and mel settings used for fine-tuning.
model_cfg = yaml.safe_load(Path('src/x_voice/configs/XVoice_Base_Stage1.yaml').read_text(encoding='utf-8'))
model_cfg['model']['tokenizer'] = 'custom'
model_cfg['model']['tokenizer_path'] = VOCAB
model_cfg['model']['arch']['languages'] = ['bg','cs','da','de','el','en','es','et','fi','fr','hr','hu','id','it','ja','ko','lt','lv','mt','nl','pl','pt','ro','ru','sk','sl','sv','th','vi','zh','bd','mn','st','ck','gr','ng']
# Download the matching Vocos model on first run.
model_cfg['model']['vocoder']['is_local'] = False
Path('/content/XVoice_Infer.yaml').write_text(yaml.safe_dump(model_cfg, sort_keys=False), encoding='utf-8')
print('Language rows:', len(model_cfg['model']['arch']['languages']))

In [ ]:
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)
cmd = [sys.executable, '-m', 'x_voice.infer.infer_cli_stage1',
       '--model_cfg', '/content/XVoice_Infer.yaml', '--ckpt_file', CHECKPOINT,
       '--vocab_file', VOCAB, '--ref_audio', REFERENCE_AUDIO, '--ref_text', REF_TEXT,
       '--ref_lang', REF_LANG, '--gen_lang', GEN_LANG, '--gen_text', GEN_TEXT,
       '--output_dir', OUTPUT_DIR, '--output_file', OUTPUT_FILE, '--sp_type', 'syllable',
       '--normalize_text', '--denoise_ref', '--loudness_norm', '--post_processing']
# X-Voice inference uses PyTorch. Disable Transformers' optional TensorFlow probe,
# which can fail in Colab when its JAX and NumPy versions are incompatible.
inference_env = os.environ.copy()
inference_env['USE_TF'] = '0'
result = subprocess.run(cmd, text=True, capture_output=True, env=inference_env)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print('--- inference error output ---')
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Inference failed with exit code {result.returncode}. See the CLI output above.')
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
print('Saved:', output_path)

In [ ]:
from IPython.display import Audio, display
display(Audio(filename=output_path))

## Save and troubleshoot

Copy the WAV into Drive with:
```python
import shutil
shutil.copy2(output_path, '/content/drive/MyDrive/xvoice/generated.wav')
```

Use the exact expanded vocabulary saved with training: its token order must match the checkpoint's enlarged input embedding. This notebook assumes the Stage 1 architecture and 24 kHz / 100-mel settings in the repo's custom fine-tune config. If you used different language IDs, edit the config cell's `languages` list to exactly match training. The language code selects a language embedding; it does not translate text or convert ordinary spelling to IPA. Supply text in the same representation used by your training data/tokenizer.

The ASR pipeline and Korean `g2pk` package are optional and load only when their features are used. Your six-language inference does not need Korean G2P; this avoids startup failure when `g2pk` is not installed. The repo also imports Transformers' ASR pipeline only if you leave the reference transcript empty for automatic transcription, and the generation cell sets `USE_TF=0` as a safeguard. If inference exits with another error, the cell prints the CLI output before raising; share that output printed above the final `RuntimeError`.